In [1]:
from agent1.classification.tfidf_classifier import TfidfIntentClassifier
from agent1.classification.semantic_classifier import SemanticIntentClassifier
from agent1.classification.ensemble import IntentClassifierEnsemble

In [ ]:
from pathlib import Path
import pandas as pd

# Notebook is in agent1/notebooks/scratch/
PROJECT_ROOT = Path.cwd().parents[1]

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Bitext-retail-ecommerce-llm-chatbot-training-dataset.parquet"
)

In [6]:
df = pd.read_parquet(
    DATA_PATH,
    columns=["instruction", "intent", "category"]
)

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["intent"]
)

In [8]:
X_train = train_df["instruction"]
X_test  = test_df["instruction"]

y_intent_train = train_df["intent"]
y_intent_test  = test_df["intent"]

In [16]:
import numpy as np
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "processed" / "embeddings"
#Loading embeddings whenever we need to use them again, without recomputing:
X_train_emb = np.load(
    EMBEDDINGS_DIR / "X_train_embeddings.npy"
)

X_test_emb = np.load(
    EMBEDDINGS_DIR / "X_test_embeddings.npy"
)

In [ ]:
#fit the tfidf model
tfidf_model=TfidfIntentClassifier()
tfidf_model.fit(X_train, y_intent_train)

#fit the semantic model
semantic_model=SemanticIntentClassifier()
semantic_model.fit(X_train_emb, y_intent_train)




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/562 [00:00<?, ?it/s]

In [12]:
# fit the ensemble model
ensemble_model = IntentClassifierEnsemble(
    tfidf_params={
        "ngram_range": (1, 2),
        "min_df": 2,
        "sublinear_tf": True,
        "max_iter": 1000,
    },
    semantic_params={
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "max_iter": 1000,
        "device": None,
    },
    tfidf_weight=0.5,
)

ensemble_model.fit(X_train, y_intent_train)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/562 [00:00<?, ?it/s]

In [13]:
#evaluation
from sklearn.metrics import accuracy_score, classification_report

y_pred_intent = tfidf_model.predict(X_test)

print(
    "tfidf accuracy:",
    accuracy_score(y_intent_test, y_pred_intent)
)

print(
    classification_report(
        y_intent_test,
        y_pred_intent
    )
)

tfidf accuracy: 0.9861869221343433
                                precision    recall  f1-score   support

                   add_product       1.00      1.00      1.00       191
                  availability       0.97      0.99      0.98       194
         availability_in_store       0.99      0.99      0.99       151
           availability_online       0.99      0.99      0.99       199
                  cancel_order       0.99      0.98      0.99       199
                change_account       0.96      1.00      0.98       197
                  change_order       0.99      1.00      1.00       192
                 close_account       0.99      0.95      0.97       199
              customer_service       1.00      0.99      0.99       198
              damaged_delivery       0.99      0.98      0.98       198
                delivery_issue       1.00      0.99      0.99       199
                 delivery_time       1.00      0.99      1.00       184
              exchange_produ

In [ ]:
y_pred_semantic = semantic_model.predict(
    X_test_emb
)

semantic_accuracy = accuracy_score(
    y_intent_test,
    y_pred_semantic
)

print(f"Semantic intent accuracy: {semantic_accuracy:.4f}")

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Semantic intent accuracy: 0.9882


In [18]:
y_pred_ensemble = ensemble_model.predict(
    X_test
)

ensemble_accuracy = accuracy_score(
    y_intent_test,
    y_pred_ensemble
)

print(f"Ensemble intent accuracy: {ensemble_accuracy:.4f}")

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Ensemble intent accuracy: 0.9910
